# ML_U4_C01 — K-means: Aprendizaje sin Etiquetas

📝 **Modalidad: Clase interactiva — sigue junto al profesor.**

**Versión:** 2025-1 | **Modificado:** 2026-05-30

---

## 📋 Mapa de la clase

| Sección | Tema | Tiempo |
|---------|------|--------|
| 1 | ¿Qué es el aprendizaje no supervisado? | 10 min |
| 2 | El algoritmo K-means: intuición y convergencia | 25 min |
| 3 | Elegir K: método del codo y silhouette | 20 min |
| 4 | Limitaciones de K-means | 15 min |
| 5 | K-means++ y variantes prácticas | 20 min |
| 6 | Ejercicio en clase | 10 min |
| 7 | Resumen y conexiones | 5 min |

---

## 📚 Prerequisitos

### 🔵 Pregrado
- Distancia euclidiana y concepto de media
- Noción de similitud / disimilitud entre puntos
- Experiencia con clasificación supervisada (unidades anteriores)

### 🟡 Doctorado
- Todo lo anterior, además:
- Modelos de mezcla gaussiana (GMM) a nivel conceptual
- Algoritmo EM: pasos E y M, función de log-verosimilitud
- Nociones de teoría de optimización: convergencia de algoritmos iterativos

---

## 🎯 Objetivos de aprendizaje

Al terminar esta clase podrás:
- Distinguir aprendizaje supervisado de no supervisado y explicar cuándo usar cada uno
- Implementar K-means desde cero y con scikit-learn
- Elegir K con el método del codo y la silueta
- Identificar situaciones donde K-means falla y proponer alternativas
- **(Doctorado)** Derivar K-means como caso límite del algoritmo EM sobre mezclas gaussianas

## ⚙️ Setup (NO MODIFICAR)

In [ ]:
# ── SETUP — NO MODIFICAR ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.datasets import make_blobs, make_moons, make_circles, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Datasets de la clase
# 1. Sintético controlado
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=0.8,
                               random_state=RANDOM_STATE)
# 2. Iris clásico
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

import sklearn
print(f"✅ Setup completo")
print(f"   numpy {np.__version__} | sklearn {sklearn.__version__}")
print(f"   Blobs: {X_blobs.shape} | Iris: {X_iris.shape}")

---
## Sección 1 — ¿Qué es el Aprendizaje No Supervisado? (10 min)

Hasta ahora cada ejemplo de entrenamiento tenía una etiqueta: spam/no spam, tumor/benigno, precio. El algoritmo aprendía a predecir esa etiqueta.

El **aprendizaje no supervisado** trabaja sin etiquetas. El objetivo ya no es predecir, sino **descubrir estructura** en los datos.

> 🧭 Cambio de pregunta: de "¿Qué es este ejemplo?" a "¿Qué patrones existen en estos datos?"

### Tareas principales del aprendizaje no supervisado

| Tarea | Pregunta | Ejemplos |
|-------|----------|----------|
| **Clustering** | ¿Qué grupos naturales existen? | Segmentación de clientes, genes similares |
| **Reducción de dimensión** | ¿Cuáles son las dimensiones más informativas? | PCA, t-SNE, UMAP |
| **Detección de anomalías** | ¿Qué puntos son inusuales? | Fraude, fallas industriales |
| **Modelos generativos** | ¿Cómo generar nuevos datos similares? | VAE, GAN |

Esta semana trabajamos **clustering**. La próxima unidad: reducción de dimensionalidad.

In [ ]:
# ━━━ MOTIVACIÓN: LA MISMA DATA, CON Y SIN ETIQUETAS ━━━
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sin etiquetas — lo que ve un algoritmo no supervisado
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1],
                c='steelblue', alpha=0.6, s=40, edgecolors='white', linewidths=0.5)
axes[0].set_title('Lo que vemos sin etiquetas\n(¿hay estructura aquí?)', fontsize=12)
axes[0].set_xlabel('Feature 1'); axes[0].set_ylabel('Feature 2')

# Con etiquetas — lo que veríamos si fueran supervisados
scatter = axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1],
                          c=y_blobs, cmap='tab10', alpha=0.7, s=40,
                          edgecolors='white', linewidths=0.5)
axes[1].set_title('Los grupos "reales" (verdad de terreno)\n→ queremos descubrirlos sin verlos', fontsize=12)
axes[1].set_xlabel('Feature 1')
plt.colorbar(scatter, ax=axes[1], label='Grupo real')

plt.suptitle('Aprendizaje No Supervisado: Descubrir Estructura sin Etiquetas',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("💡 El clustering intenta recuperar la estructura de la derecha usando solo los datos de la izquierda")

---
## Sección 2 — El Algoritmo K-means (25 min)

K-means es el algoritmo de clustering más utilizado. La idea es simple:

1. **Inicializar** K centroides aleatoriamente
2. **Asignar** cada punto al centroide más cercano
3. **Actualizar** cada centroide como la media de los puntos asignados
4. **Repetir** hasta convergencia

**Función objetivo** (inercia o WCSS — Within-Cluster Sum of Squares):
$$J = \sum_{k=1}^{K} \sum_{x_i \in C_k} \|x_i - \mu_k\|^2$$

Donde $\mu_k$ es el centroide del cluster $C_k$. K-means minimiza esta función.

In [ ]:
# ━━━ ANIMACIÓN PASO A PASO DE K-MEANS (4 iteraciones) ━━━
from sklearn.cluster import KMeans

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()
colors = ['#E74C3C', '#2E86C1', '#27AE60', '#8E44AD']

# Inicialización manual (misma seed para reproducibilidad)
rng = np.random.RandomState(7)
centers_init = X_blobs[rng.choice(len(X_blobs), 4, replace=False)]
centers = centers_init.copy()

for step in range(6):
    ax = axes[step]
    # Paso E: asignar puntos
    dists = np.linalg.norm(X_blobs[:, None, :] - centers[None, :, :], axis=2)
    labels = dists.argmin(axis=1)

    # Graficar puntos
    for k in range(4):
        mask = labels == k
        ax.scatter(X_blobs[mask, 0], X_blobs[mask, 1],
                   c=colors[k], alpha=0.5, s=30, edgecolors='white', linewidths=0.3)

    # Graficar centroides
    ax.scatter(centers[:, 0], centers[:, 1],
               c=colors, marker='*', s=300, edgecolors='black', linewidths=1.5, zorder=5)

    inertia = sum(np.sum((X_blobs[labels == k] - centers[k])**2) for k in range(4))
    ax.set_title(f'Iteración {step} | Inercia = {inertia:.1f}', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

    # Paso M: actualizar centroides
    new_centers = np.array([X_blobs[labels == k].mean(axis=0) if (labels == k).sum() > 0
                             else centers[k] for k in range(4)])
    centers = new_centers

plt.suptitle('K-means: Convergencia Paso a Paso (K=4)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("💡 Observa cómo la inercia disminuye monótonamente hasta estabilizarse")

In [ ]:
# ━━━ K-MEANS CON SCIKIT-LEARN — APLICACIÓN A IRIS ━━━
# Iris tiene 3 especies → K=3 es un candidato natural
scaler = StandardScaler()
X_iris_sc = scaler.fit_transform(X_iris)

kmeans_iris = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
labels_iris = kmeans_iris.fit_predict(X_iris_sc)

# Visualizar con PCA 2D para poder graficar las 4 features
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_iris_2d = pca.fit_transform(X_iris_sc)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Etiquetas reales
for k, name in enumerate(iris.target_names):
    mask = y_iris == k
    axes[0].scatter(X_iris_2d[mask, 0], X_iris_2d[mask, 1],
                    label=name, s=50, alpha=0.7, edgecolors='white', linewidths=0.5)
axes[0].set_title('Etiquetas reales de Iris (3 especies)', fontsize=11)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var.)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var.)')
axes[0].legend()

# Clusters K-means
for k in range(3):
    mask = labels_iris == k
    axes[1].scatter(X_iris_2d[mask, 0], X_iris_2d[mask, 1],
                    label=f'Cluster {k}', s=50, alpha=0.7, edgecolors='white', linewidths=0.5)
centers_2d = pca.transform(kmeans_iris.cluster_centers_)
axes[1].scatter(centers_2d[:, 0], centers_2d[:, 1],
                marker='*', s=300, c='black', zorder=5, label='Centroides')
axes[1].set_title(f'K-means K=3 | Inercia={kmeans_iris.inertia_:.1f}', fontsize=11)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var.)')
axes[1].legend()

ari = adjusted_rand_score(y_iris, labels_iris)
print(f"Adjusted Rand Index (ARI): {ari:.4f}  [0=aleatorio, 1=perfecto]")
print("💡 ARI mide qué tan bien los clusters recuperan las clases reales")
plt.tight_layout(); plt.show()

---
## Sección 3 — Elegir K: Método del Codo y Silhouette (20 min)

El número de clusters K es el **hiperparámetro más importante** de K-means.
A diferencia de la clasificación supervisada, no hay una métrica de validación directa (no hay etiquetas).
Se usan indicadores internos:

### 3.1 — Método del Codo (Elbow Method)

Graficar la **inercia** (WCSS) en función de K. La inercia siempre cae al aumentar K.
Se busca el punto donde la caída se aplana — el "codo".

### 3.2 — Silhouette Score

Para cada punto $i$, la silueta mide qué tan bien está en su cluster vs. el cluster vecino:
$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

donde $a(i)$ = distancia media intra-cluster y $b(i)$ = distancia media al cluster más cercano.

$s(i) \in [-1, 1]$: cercano a 1 → bien agrupado; cercano a 0 → frontera; negativo → mal asignado.

In [ ]:
# ━━━ MÉTODO DEL CODO + SILHOUETTE SOBRE BLOBS ━━━
K_range = range(2, 10)
inertias, silhouettes = [], []

X_blobs_sc = StandardScaler().fit_transform(X_blobs)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels_k = km.fit_predict(X_blobs_sc)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_blobs_sc, labels_k))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Codo
axes[0].plot(K_range, inertias, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].axvline(4, color='tomato', linestyle='--', alpha=0.7, label='Codo en K=4')
axes[0].set_xlabel('Número de clusters K'); axes[0].set_ylabel('Inercia (WCSS)')
axes[0].set_title('Método del Codo'); axes[0].legend()

# Silhouette
best_sil_k = list(K_range)[int(np.argmax(silhouettes))]
axes[1].plot(K_range, silhouettes, 'o-', color='seagreen', linewidth=2, markersize=8)
axes[1].axvline(best_sil_k, color='tomato', linestyle='--', alpha=0.7,
                label=f'Mejor K={best_sil_k} (sil={max(silhouettes):.3f})')
axes[1].set_xlabel('Número de clusters K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score'); axes[1].legend()

plt.suptitle('Selección de K — Dataset Blobs (K verdadero = 4)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"Ambos métodos coinciden: K óptimo = {best_sil_k} (K verdadero = 4) ✅")

In [ ]:
# ━━━ DIAGRAMA DE SILUETA POR CLUSTER ━━━
from sklearn.metrics import silhouette_samples

km_final = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
labels_final = km_final.fit_predict(X_blobs_sc)
sil_vals = silhouette_samples(X_blobs_sc, labels_final)

fig, ax = plt.subplots(figsize=(9, 5))
y_lower = 10
cluster_colors = plt.cm.tab10(np.linspace(0, 0.4, 4))

for k in range(4):
    sil_k = np.sort(sil_vals[labels_final == k])
    size_k = len(sil_k)
    y_upper = y_lower + size_k
    ax.barh(range(y_lower, y_upper), sil_k, height=1.0,
            color=cluster_colors[k], edgecolor='none', alpha=0.85,
            label=f'Cluster {k} (n={size_k}, med={np.median(sil_k):.2f})')
    ax.text(-0.05, y_lower + size_k / 2, str(k), ha='right', va='center', fontsize=10)
    y_lower = y_upper + 10

sil_avg = silhouette_score(X_blobs_sc, labels_final)
ax.axvline(sil_avg, color='red', linestyle='--', alpha=0.7,
           label=f'Promedio = {sil_avg:.3f}')
ax.set_xlabel('Silhouette coefficient'); ax.set_title('Diagrama de Silueta — K=4')
ax.legend(loc='lower right', fontsize=8)
ax.set_yticks([])
plt.tight_layout(); plt.show()
print("💡 Clusters anchos y uniformes → buena separación. Picos negativos → puntos mal asignados")

---
## Sección 4 — Limitaciones de K-means (15 min)

K-means asume que los clusters son:
- **Esféricos** (isótropos): equidistantes en todas las direcciones desde el centroide
- **Convexos**: sin formas cóncavas o en anillo
- **De tamaño similar**: la inercia favorece clusters balanceados

Cuando estas suposiciones no se cumplen, K-means falla sistemáticamente.

In [ ]:
# ━━━ CASOS DONDE K-MEANS FALLA ━━━
from sklearn.datasets import make_moons, make_circles

# Datos problemáticos
X_moons, y_moons = make_moons(n_samples=300, noise=0.07, random_state=RANDOM_STATE)
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=RANDOM_STATE)

# Clusters de tamaño muy desigual
X_big = np.random.RandomState(0).randn(400, 2) * 0.5 + [0, 0]
X_small = np.random.RandomState(1).randn(50, 2) * 0.1 + [3, 3]
X_unequal = np.vstack([X_big, X_small])
y_unequal = np.array([0]*400 + [1]*50)

datasets = [
    (X_moons,   y_moons,   'Lunas (no convexo)'),
    (X_circles, y_circles, 'Círculos (anillos)'),
    (X_unequal, y_unequal, 'Tamaños desiguales'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for col, (X, y_true, title) in enumerate(datasets):
    X_sc = StandardScaler().fit_transform(X)
    km = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10)
    pred = km.fit_predict(X_sc)

    # Verdad real
    axes[0, col].scatter(X[:, 0], X[:, 1], c=y_true, cmap='RdBu', s=30, alpha=0.7,
                         edgecolors='white', linewidths=0.3)
    axes[0, col].set_title(f'{title}\nVerdad real', fontsize=10)
    axes[0, col].set_xticks([]); axes[0, col].set_yticks([])

    # K-means
    ari = adjusted_rand_score(y_true, pred)
    axes[1, col].scatter(X[:, 0], X[:, 1], c=pred, cmap='RdBu', s=30, alpha=0.7,
                         edgecolors='white', linewidths=0.3)
    axes[1, col].scatter(km.cluster_centers_[:, 0] * X[:, 0].std() + X[:, 0].mean(),
                         km.cluster_centers_[:, 1] * X[:, 1].std() + X[:, 1].mean(),
                         marker='*', s=300, c='black', zorder=5)
    axes[1, col].set_title(f'K-means K=2 | ARI={ari:.2f}', fontsize=10,
                            color='tomato' if ari < 0.5 else 'seagreen')
    axes[1, col].set_xticks([]); axes[1, col].set_yticks([])

axes[0, 0].set_ylabel('Verdad real', fontsize=11)
axes[1, 0].set_ylabel('K-means', fontsize=11)
plt.suptitle('Limitaciones de K-means: Supuestos que Viola', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print("💡 ARI bajo (rojo) indica que K-means falla en recuperar la estructura real")

**Alternativas para cada caso:**

| Problema | Alternativa recomendada |
|----------|------------------------|
| Clusters no convexos (lunas, anillos) | DBSCAN, Spectral Clustering |
| Clusters elípticos | GMM (Gaussian Mixture Models) |
| Tamaños muy desiguales | DBSCAN, GMM con covarianza libre |
| Alta dimensionalidad | Reducción primero (PCA), luego K-means |

---
## Sección 5 — K-means++ y Variantes Prácticas (20 min)

### 5.1 — El Problema de la Inicialización

K-means puede converger a mínimos locales malos dependiendo de la inicialización.
**K-means++** (Arthur & Vassilvitskii, 2007) resuelve esto con una inicialización inteligente:

1. Elegir el primer centroide uniformemente al azar
2. Elegir cada centroide siguiente con probabilidad proporcional a $d(x)^2$: los puntos más alejados de los centroides existentes tienen más probabilidad de ser elegidos
3. Repetir hasta tener K centroides

**Garantía:** K-means++ encuentra una solución que, en expectativa, es $O(\log K)$ veces peor que el óptimo global. Es el default en scikit-learn (`init='k-means++'`).

In [ ]:
# ━━━ COMPARACIÓN: RANDOM vs K-MEANS++ ━━━
n_runs = 20
inertias_random = []
inertias_kpp = []

for seed in range(n_runs):
    km_rand = KMeans(n_clusters=4, init='random', n_init=1, random_state=seed)
    km_rand.fit(X_blobs_sc)
    inertias_random.append(km_rand.inertia_)

    km_kpp = KMeans(n_clusters=4, init='k-means++', n_init=1, random_state=seed)
    km_kpp.fit(X_blobs_sc)
    inertias_kpp.append(km_kpp.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(inertias_random, bins=10, color='tomato', alpha=0.7, label='Random init')
axes[0].hist(inertias_kpp, bins=10, color='steelblue', alpha=0.7, label='K-means++')
axes[0].axvline(np.mean(inertias_random), color='tomato', linestyle='--', linewidth=2)
axes[0].axvline(np.mean(inertias_kpp), color='steelblue', linestyle='--', linewidth=2)
axes[0].set_xlabel('Inercia final'); axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Inercias (20 corridas, n_init=1)')
axes[0].legend()

axes[1].boxplot([inertias_random, inertias_kpp], labels=['Random', 'K-means++'],
                patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='tomato', linewidth=2))
axes[1].set_ylabel('Inercia final'); axes[1].set_title('Boxplot de Inercias')

plt.suptitle('K-means++ vs Inicialización Aleatoria', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"Random init  — media: {np.mean(inertias_random):.2f} ± {np.std(inertias_random):.2f}")
print(f"K-means++   — media: {np.mean(inertias_kpp):.2f} ± {np.std(inertias_kpp):.2f}")
print("\n💡 K-means++ es más consistente y alcanza mejores mínimos en promedio")

In [ ]:
# ━━━ MINI-BATCH K-MEANS: PARA DATOS GRANDES ━━━
import time

# Dataset grande sintético
X_large, _ = make_blobs(n_samples=50_000, centers=10, random_state=RANDOM_STATE)
X_large_sc = StandardScaler().fit_transform(X_large)

t0 = time.time()
km_full = KMeans(n_clusters=10, random_state=RANDOM_STATE, n_init=3)
km_full.fit(X_large_sc)
t_full = time.time() - t0

t0 = time.time()
km_mini = MiniBatchKMeans(n_clusters=10, random_state=RANDOM_STATE, n_init=3, batch_size=1024)
km_mini.fit(X_large_sc)
t_mini = time.time() - t0

print(f"Dataset: {X_large_sc.shape[0]:,} puntos, K=10")
print(f"K-means full   — inercia: {km_full.inertia_:.1f}  | tiempo: {t_full:.3f}s")
print(f"MiniBatch K-means — inercia: {km_mini.inertia_:.1f} | tiempo: {t_mini:.3f}s")
print(f"Ratio de velocidad: {t_full/t_mini:.1f}x más rápido")
print("\n💡 MiniBatchKMeans sacrifica un poco de calidad por velocidad → útil para millones de puntos")

---
## Sección 6 — Ejercicio en Clase (10 min)

### Parte A — Sin computador (5 min) 🖊️

Tienes 6 puntos en 1D: $\{1, 2, 3, 8, 9, 10\}$ y quieres K-means con K=2.

**1.** Los centroides iniciales son $\mu_1 = 2$, $\mu_2 = 9$. Aplica una iteración de K-means:
   - ¿Cómo se asigna cada punto?
   - ¿Cuáles son los nuevos centroides?

**2.** ¿Converge en una segunda iteración? ¿Cuál es la inercia final?

**3.** Si los centroides iniciales hubieran sido $\mu_1 = 1$, $\mu_2 = 2$, ¿a qué solución converge? ¿Es un mínimo local o global?

*Escribe tu respuesta aquí antes de ejecutar el código:*

In [ ]:
# ━━━ VERIFICACIÓN PARTE A ━━━
puntos = np.array([1., 2., 3., 8., 9., 10.])
mu = np.array([2., 9.])

print("Iteración 1:")
for x in puntos:
    d1, d2 = abs(x - mu[0]), abs(x - mu[1])
    asignado = 0 if d1 <= d2 else 1
    print(f"  x={x:.0f}: d(μ1)={d1}, d(μ2)={d2} → Cluster {asignado}")

C1 = puntos[puntos < 5.5]
C2 = puntos[puntos >= 5.5]
mu_nuevo = np.array([C1.mean(), C2.mean()])
print(f"\nNuevos centroides: μ1={mu_nuevo[0]:.2f}, μ2={mu_nuevo[1]:.2f}")

inercia = sum((x - mu_nuevo[0])**2 for x in C1) + sum((x - mu_nuevo[1])**2 for x in C2)
print(f"Inercia final: {inercia:.2f}")
print("\n💡 Con μ1=1, μ2=2 → ambos centroides en C1=[1,2,3] → mínimo LOCAL, no global")

---
## Sección 7 — Resumen y Conexiones (5 min)

### 📊 Tabla resumen — K-means

| Aspecto | Detalle |
|---------|--------|
| **Objetivo** | Minimizar inercia (WCSS) |
| **Complejidad** | $O(n \cdot K \cdot d \cdot T)$ por iteración |
| **Supuestos** | Clusters esféricos, convexos, de tamaño similar |
| **Hiperparámetro clave** | K (usar codo + silhouette) |
| **Siempre hacer** | Normalizar datos + usar K-means++ + n_init>1 |
| **Cuándo falla** | Clusters no esféricos, densidades distintas, outliers |

### 🔗 Próxima clase: Clustering Jerárquico

K-means requiere especificar K de antemano. El **clustering jerárquico** construye
un árbol completo de clusters (dendrograma) que permite elegir K después de ver la estructura.

---

### 📚 Bibliografía

#### Pregrado
- Géron, A. (2022). *Hands-On ML* (3ª ed.). Cap. 9: Unsupervised Learning Techniques.
- sklearn documentation: [KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)

#### Doctorado / Investigación
- Arthur, D., & Vassilvitskii, S. (2007). k-means++. *SODA 2007*.
- Bishop, C. M. (2006). *PRML*. Cap. 9: Mixture Models and EM.
- Tibshirani, R. et al. (2001). Gap Statistic. *JRSS-B*.
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning*. Cap. 14.